# Classificazione
(1 esercizio - 2 esempi)

## 1: Implementazione GradCAM manuale con hook di PyTorch


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
import cv2

# Trasformazioni
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

# Caricamento del dataset
train_dataset = datasets.FashionMNIST(root='data', train=True, download=True, transform=transform)
train_len = int(0.8 * len(train_dataset))
valid_len = len(train_dataset) - train_len
train_dataset, valid_dataset = random_split(train_dataset, [train_len, valid_len])

test_dataset = datasets.FashionMNIST(root='data', train=False, download=True, transform=transform)
test_dataset, _ = random_split(test_dataset, [20, len(test_dataset) - 20])      # Solo 20 immagini per il test

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 32 * 7 * 7)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
class ManualGradCAM:
  """Implementazione minimale di Grad-CAM con hook PyTorch."""

  def __init__(self, model: nn.Module, target_layer: nn.Module):
    self.model = model
    self.target_layer = target_layer
    self.gradients = None         # Gradiente backpropagato rispetto al layer target
    self.activations = None       # Attivazioni forward del layer target
    self._register_hooks()

  def _register_hooks(self):
    # Hook di forward: viene eseguito automaticamente quando il layer target produce il suo output; memorizziamo le attivazioni per riutilizzarle dopo.
    def forward_hook(_, __, output):
      self.activations = output.detach()            # Salva feature maps prodotte dal layer

    # Hook di backward: viene chiamato durante la retropropagazione. grad_output[0] è il gradiente della loss rispetto all'output del layer target, il "peso" di Grad-CAM.
    def backward_hook(_, grad_input, grad_output):
      self.gradients = grad_output[0].detach()      # Salva gradiente rispetto all'output del layer

    # Registriamo i due hook sul layer desiderato: così, al prossimo forward/backward, avremo automaticamente sia attivazioni sia gradienti senza modificare il modello.
    self.target_layer.register_forward_hook(forward_hook)         # Hook per catturare attivazioni
    self.target_layer.register_full_backward_hook(backward_hook)  # Hook per catturare gradienti

  def generate(self, input_tensor: torch.Tensor, class_idx: int | None = None) -> np.ndarray:
    self.model.zero_grad()                # Reset gradiente prima del backward
    scores = self.model(input_tensor)     # Forward completo

    if class_idx is None:
      class_idx = int(scores.argmax(dim=1).item())    # Usa la classe predetta
    elif torch.is_tensor(class_idx):
      class_idx = int(class_idx.item())

    score = scores[:, class_idx]          # Logit della classe target
    # Non ci interessa il gradiente sul logit stesso: l'esecuzione di backward attiva gli hook sul layer target, che salvano gradiente/attivazioni senza bisogno di variabili extra
    score.sum().backward()                # Backprop per raccogliere gradienti sul layer registrato

    if self.gradients is None or self.activations is None:
      raise RuntimeError("GradCAM hooks non hanno catturato gradiente/attivazioni.")

    weights = self.gradients.mean(dim=(2, 3), keepdim=True)                                                   # Pesi global average pooling
    cam = torch.relu((weights * self.activations).sum(dim=1, keepdim=True))                                   # Combinazione lineare + ReLU
    cam = nn.functional.interpolate(cam, size=input_tensor.shape[-2:], mode="bilinear", align_corners=False)  # Resize alla dimensione immagine

    heatmap = cam.squeeze().cpu().numpy()
    heatmap -= heatmap.min()
    heatmap /= (heatmap.max() + 1e-8)
    return heatmap

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, loader, optimizer, criterion, device):
  model.train()
  total_loss, correct = 0, 0
  for data, target in loader:
    data, target = data.to(device), target.to(device)
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, target)
    loss.backward()

    optimizer.step()
    total_loss += loss.item()
    correct += (output.argmax(1) == target).sum().item()
    break
  return total_loss / len(loader), correct / len(loader.dataset)

def validate(model, loader, criterion, device):
  model.eval()
  total_loss, correct = 0, 0
  with torch.no_grad():
    for data, target in loader:
      data, target = data.to(device), target.to(device)
      output = model(data)
      loss = criterion(output, target)

      total_loss += loss.item()
      correct += (output.argmax(1) == target).sum().item()
      break
  return total_loss / len(loader), correct / len(loader.dataset)

for epoch in range(5):
  train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
  valid_loss, valid_acc = validate(model, valid_loader, criterion, device)
  print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}, Valid Loss: {valid_loss:.4f}, Valid Acc: {valid_acc:.2f}")

In [ ]:
def overlay_heatmap(image_tensor: torch.Tensor, heatmap: np.ndarray) -> np.ndarray:
  image_np = image_tensor.squeeze().cpu().numpy()
  image_np = (image_np - image_np.min()) / (image_np.max() - image_np.min() + 1e-8)
  image_np = np.uint8(255 * image_np)
  image_rgb = cv2.cvtColor(image_np, cv2.COLOR_GRAY2RGB)                        # Replica su 3 canali per visualizzare meglio

  heatmap = cv2.resize(heatmap, (image_rgb.shape[1], image_rgb.shape[0]))       # Porta Grad-CAM alla stessa risoluzione
  heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)  # Colormap per evidenziare le zone calde
  overlay = cv2.addWeighted(image_rgb, 0.5, heatmap_color, 0.5, 0)              # Fusione immagine/origine e heatmap
  return overlay

cam_extractor = ManualGradCAM(model, target_layer=model.conv2)                  # Usa l'ultimo layer convoluzionale come target

for i, (image, label) in enumerate(test_loader):
  image, label = image.to(device), label.to(device)
  output = model(image)
  pred_class = output.argmax(dim=1)
  heatmap = cam_extractor.generate(image, pred_class)                               # Calcola Grad-CAM per la classe predetta
  overlay = overlay_heatmap(image.cpu(), heatmap)                                   # Combina heatmap e immagine originale
  cv2.imwrite(f"gradcam_manual_{i}.png", cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))  # Salva in formato compatibile con OpenCV

print("Grad-CAM manuale completato e immagini salvate.")

## 2: Implementazione di GradCAM con torchcam (GradCAM e overlay_mask)

In [ ]:
!pip install torchcam

import matplotlib.pyplot as plt

from torchcam.methods import GradCAM
from torchcam.utils import overlay_mask

# Trasformazioni - Caricamento del dataset
# Definizione di una CNN semplice
# Inizializzazione del modello - Addestramento - Funz. di validazione
# Addestramento per 5 epoche

# Creazione del Grad-CAM sul layer desiderato
cam_extractor = GradCAM(model, target_layer="conv2")

# Applicazione di Grad-CAM sul test set
for i, (image, label) in enumerate(test_loader):
  image, label = image.to(device), label.to(device)

  output = model(image)                           # Forward pass
  pred_class = output.argmax(dim=1).item()

  # Calcolo della heatmap Grad-CAM
  cam = cam_extractor(pred_class, output)         # Heatmap

  img = image.squeeze(0).cpu()                    # Sovrapposizione heatmap sull'immagine originale
  img_rgb = img.repeat(3, 1, 1)                   # Ripeti il canale per creare un'immagine RGB
  result = overlay_mask(transforms.ToPILImage()(img_rgb), transforms.ToPILImage()(cam[0].squeeze(0)), alpha=0.5)

  result.save(f"gradcam_result_{i}.png")          # Salvare l'immagine risultante

print("Grad-CAM completato e immagini salvate.")

## 3: Addestrare la rete ResNet18 sul dataset FashionMNIST e usare GradCAM per verificare cosa la rete vede per la classificazione

In [ ]:
!pip install torchcam
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import cv2
import numpy as np
from torchcam.methods import GradCAM
from torchcam.utils import overlay_mask

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Dataset
train_dataset = datasets.FashionMNIST(root='data', train=True, download=True, transform=transform)
train_len = int(0.8 * len(train_dataset))
valid_len = len(train_dataset) - train_len
train_dataset, valid_dataset = random_split(train_dataset, [train_len, valid_len])

test_dataset = datasets.FashionMNIST(root='data', train=False, download=True, transform=transform)
test_dataset, _ = random_split(test_dataset, [20, len(test_dataset) - 20])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# --- ResNet18 ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=None)

# Modifica primo layer (1 canale)
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

# Cambia output finale (10 classi)
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, loader, optimizer, criterion, device):
  model.train()
  total_loss, correct = 0, 0
  for data, target in loader:
    data, target = data.to(device), target.to(device)
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, target)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    correct += (output.argmax(1) == target).sum().item()
    break
  return total_loss / len(loader), correct / len(loader.dataset)

def validate(model, loader, criterion, device):
  model.eval()
  total_loss, correct = 0, 0
  with torch.no_grad():
    for data, target in loader:
      data, target = data.to(device), target.to(device)
      output = model(data)
      loss = criterion(output, target)
      total_loss += loss.item()
      correct += (output.argmax(1) == target).sum().item()
      break
    return total_loss / len(loader), correct / len(loader.dataset)

for epoch in range(5):
  train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
  valid_loss, valid_acc = validate(model, valid_loader, criterion, device)
  print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}, Valid Loss: {valid_loss:.4f}, Valid Acc: {valid_acc:.2f}")

cam_extractor = GradCAM(model, target_layer="layer4")

for i, (image, label) in enumerate(test_loader):
    image = image.to(device)

    output = model(image)
    pred_class = output.argmax(dim=1).item()

    cams = cam_extractor(pred_class, output)

    cam = cams[0].squeeze().cpu().numpy()

    cam = (cam - cam.min()) / (cam.max() - cam.min())
    cam = (cam * 255).astype(np.uint8)

    cam = cv2.resize(cam, (224, 224))

    heatmap = cv2.applyColorMap(cam, cv2.COLORMAP_JET)

    img = image.squeeze().cpu().numpy()
    img = (img - img.min()) / (img.max() - img.min())
    img = (img * 255).astype(np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    img = cv2.resize(img, (224, 224))

    result = cv2.addWeighted(img, 0.5, heatmap, 0.5, 0)

    cv2.imwrite(f"gradcam_cv2_{i}.png", result)

print("Heatmap Grad-CAM salvate con OpenCV ✅")

Epoch 1: Train Loss: 0.0032, Train Acc: 0.00, Valid Loss: 0.0519, Valid Acc: 0.00
Epoch 2: Train Loss: 0.0032, Train Acc: 0.00, Valid Loss: 0.0278, Valid Acc: 0.00
Epoch 3: Train Loss: 0.0024, Train Acc: 0.00, Valid Loss: 0.0190, Valid Acc: 0.00
Epoch 4: Train Loss: 0.0019, Train Acc: 0.00, Valid Loss: 0.0252, Valid Acc: 0.00
Epoch 5: Train Loss: 0.0021, Train Acc: 0.00, Valid Loss: 0.0435, Valid Acc: 0.00
Heatmap Grad-CAM salvate con OpenCV ✅
